# Step 4A: `bio_capture_loop.py` Walkthrough

This notebook is a guided tour of the Phase 1 edge capture loop. It is written for a new student who wants to understand what happens between a raw field recording and a row in the edge database.

The production script is [`edge_node_mock/src/bio_capture_loop.py`](../edge_node_mock/src/bio_capture_loop.py). Here we call the same functions in small pieces so each stage is visible.

For repeatability, this notebook uses the bundled teaching clip [`notebooks/example_audio/example1_120s_petrel.wav`](example_audio/example1_120s_petrel.wav). The clip is roughly 120 seconds long and contains wind/sea noise plus two grey-faced petrel calls.

By the end you should be able to explain:

- How a raw `.wav` file becomes 15-second buffers.
- Why Perch sees three 5-second windows for each buffer.
- What the model logits and embeddings look like.
- How configured noise and biological labels drive the first-pass gate.
- How long inference takes for the full teaching clip.
- How retained audio is written as `.flac`.
- How the same objects are inserted into the edge SQLite schema.

Teacher note: the first model call may take a little while because TensorFlow loads and warms up the Perch CPU model. TensorFlow may also print CUDA warnings on machines without GPU support; for this notebook that is expected.

## 1. Project Setup

Run this notebook from the repository environment, using the root `.venv` kernel if possible. The setup cell makes imports work even if Jupyter starts from the `notebooks/` directory.

In [ ]:
from pathlib import Path
import copy
import json
import sqlite3
import sys
import time

import numpy as np
import pandas as pd
import soundfile as sf
import yaml
from IPython.display import Audio, display

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "edge_node_mock").exists():
    REPO_ROOT = Path.cwd().parent

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

pd.set_option("display.max_colwidth", 120)
print("Repository root:", REPO_ROOT)

In [ ]:
from mock_common.config import load_config
from edge_node_mock.src.bio_capture_loop import (
    build_label_index,
    decide_buffer,
    insert_buffer_event,
    iter_audio_buffers,
    load_sqlite_vec,
    run_perch_inference,
    save_retained_audio,
    score_frames,
)
from edge_node_mock.src.init_edge_db import init_edge_db
from edge_node_mock.src.inspect_perch_model import (
    _load_model,
    load_nz_bird_labels,
    load_perch_labels,
    make_perch_windows,
)

## 2. Load The YAML Config, Then Point It At The Example Clip

The capture loop is deliberately configuration-driven. Hardware-like settings, file paths, thresholds, and label groups live in YAML rather than being scattered through code.

For teaching, we load the normal edge config and then override only the raw audio location in memory. This means the notebook uses the bundled example clip without changing your local config files.

In [ ]:
CONFIG_PATH = REPO_ROOT / "edge_node_mock" / "config" / "edge_config.local.yaml"
if not CONFIG_PATH.exists():
    CONFIG_PATH = REPO_ROOT / "edge_node_mock" / "config" / "edge_config.example.yaml"

base_config = load_config(CONFIG_PATH)

EXAMPLE_AUDIO_PATH = REPO_ROOT / "notebooks" / "example_audio" / "example1_120s_petrel.wav"
if not EXAMPLE_AUDIO_PATH.exists():
    raise FileNotFoundError(f"Missing teaching clip: {EXAMPLE_AUDIO_PATH}")

config = copy.deepcopy(base_config)
config["raw_audio_mount"] = str(EXAMPLE_AUDIO_PATH.parent)
config["raw_audio_glob"] = EXAMPLE_AUDIO_PATH.name
config["retained_audio_dir"] = str(REPO_ROOT / "edge_node_mock" / "data" / "notebook_retained_audio")

important_keys = [
    "device_id",
    "raw_audio_mount",
    "raw_audio_glob",
    "perch_sample_rate",
    "embedding_dim",
    "bio_threshold",
    "noise_threshold",
    "validation_sample_interval",
]
for key in important_keys:
    print(f"{key}: {config[key]}")

In [ ]:
info = sf.info(EXAMPLE_AUDIO_PATH)
clip_duration = float(info.duration)
expected_buffers = int(np.ceil(clip_duration / 15.0))
padded_seconds = expected_buffers * 15.0 - clip_duration

print("Example audio:", EXAMPLE_AUDIO_PATH)
print("Sample rate:", info.samplerate)
print("Channels:", info.channels)
print("Duration seconds:", round(clip_duration, 3))
print("15-second buffers needed:", expected_buffers)
print("Silence padding needed for the last buffer:", round(padded_seconds, 3), "seconds")

# A handy playback widget for students. It can be skipped if the notebook UI does not render audio.
display(Audio(filename=str(EXAMPLE_AUDIO_PATH)))

In [ ]:
print("Noise labels:")
for label in config["noise_labels"]:
    print("  -", label)

print("\nBiological labels:")
for label in config["biological_labels"]:
    print("  -", label)

## 3. Load Labels And Build Lookup Tables

Perch returns a large vector of logits. A logit is a raw model score. To interpret a logit position, we need to map a numeric index back to a label name.

`build_label_index()` turns the label names in YAML into numeric indexes. This is one of the simplest and most important safety checks in the pipeline: if a label is misspelled, the script should fail early.

In [ ]:
perch_labels = load_perch_labels(REPO_ROOT / config["perch_label_path"])
nz_labels = load_nz_bird_labels(REPO_ROOT / config["nz_bird_label_path"], perch_labels)

noise_label_indexes = build_label_index(
    perch_labels,
    config["noise_labels"],
    group_name="noise_labels",
)
bio_label_indexes = build_label_index(
    perch_labels,
    config["biological_labels"],
    group_name="biological_labels",
)

print("Perch label count:", len(perch_labels))
print("NZ bird subset count:", len(nz_labels))
print("Noise indexes:", noise_label_indexes)
print("Biological indexes:", bio_label_indexes)

## 4. Stream The Full Teaching Clip Into 15-Second Buffers

`iter_audio_buffers()` is the fake microphone for Phase 1. Instead of reading from hardware, it reads `.wav` files from the configured data directory.

For each 15-second chunk it:

1. Reads a block from disk.
2. Converts stereo or mono source audio into mono `float32`.
3. Leaves 32 kHz audio alone, or downsamples 48 kHz to 32 kHz.
4. Yields an `AudioBuffer` object for inference and database writing.

The teaching clip is just under 120 seconds, so this notebook uses `include_partial=True`. That pads the last tiny shortfall with silence and lets us process the full clip as eight 15-second buffers. The production default remains `include_partial=False`.

In [ ]:
buffers = list(iter_audio_buffers(config, include_partial=True))

buffer_rows = []
for buffer in buffers:
    buffer_rows.append(
        {
            "buffer_index": buffer.file_buffer_index,
            "timestamp_utc": buffer.timestamp_utc.isoformat(),
            "source_file": buffer.source_file.name,
            "source_sample_rate": buffer.source_sample_rate,
            "perch_sample_rate": buffer.perch_sample_rate,
            "samples": len(buffer.perch_audio),
            "duration_seconds": len(buffer.perch_audio) / buffer.perch_sample_rate,
            "peak_amplitude": float(np.max(np.abs(buffer.source_audio))),
        }
    )

pd.DataFrame(buffer_rows)

The raw clip is longer than one model input. The capture loop treats it as a stream of 15-second buffers.

Perch itself expects 5-second windows at 32 kHz. A 15-second buffer therefore becomes three model rows of 160,000 samples each.

In [ ]:
first_buffer = buffers[0]
windows = make_perch_windows(first_buffer.perch_audio, first_buffer.perch_sample_rate)

print("Window array shape:", windows.shape)
print("Window duration seconds:", windows.shape[1] / first_buffer.perch_sample_rate)
print("Frame count per buffer:", windows.shape[0])
print("Total Perch frames for this clip:", len(buffers) * windows.shape[0])

## 5. Load The Perch Model

`run_perch_inference()` loads the serving signature and asks Perch for two outputs we care about in Phase 1:

- `logits`: label scores, one row per 5-second frame.
- `embeddings`: compact 1536-dimensional vectors, one row per 5-second frame.

These embeddings are what we store in SQLite for later search and analysis.

In [ ]:
model_load_start = time.perf_counter()
model, model_source, model_ref = _load_model(config)
model_load_seconds = time.perf_counter() - model_load_start

print("Model source:", model_source)
print("Model reference:", model_ref)
print("Model load seconds:", round(model_load_seconds, 3))

## 6. Run Inference Across The Full 120-Second Clip

This cell processes every 15-second buffer from the teaching clip. It times each buffer independently and also records the total inference time.

The first buffer may be slower because TensorFlow compiles/warms up the model. That is useful to notice: embedded systems often have a difference between first-run latency and steady-state latency.

In [ ]:
results = []
inference_rows = []
dropped_buffer_count = 0
full_start = time.perf_counter()

for buffer in buffers:
    infer_start = time.perf_counter()
    logits, embeddings = run_perch_inference(model, buffer.perch_audio, buffer.perch_sample_rate)
    infer_seconds = time.perf_counter() - infer_start

    frame_scores = score_frames(
        logits,
        perch_labels=perch_labels,
        noise_label_indexes=noise_label_indexes,
        bio_label_indexes=bio_label_indexes,
        nz_label_indexes=nz_labels,
    )
    decision = decide_buffer(
        frame_scores,
        bio_threshold=float(config["bio_threshold"]),
        noise_threshold=float(config["noise_threshold"]),
        validation_sample_interval=int(config["validation_sample_interval"]),
        dropped_buffer_count=dropped_buffer_count,
    )

    if decision.retention_reason == "dropped":
        dropped_buffer_count += 1

    audio_seconds = len(buffer.perch_audio) / buffer.perch_sample_rate
    results.append(
        {
            "buffer": buffer,
            "logits": logits,
            "embeddings": embeddings,
            "frame_scores": frame_scores,
            "decision": decision,
            "infer_seconds": infer_seconds,
        }
    )
    inference_rows.append(
        {
            "buffer_index": buffer.file_buffer_index,
            "audio_seconds": audio_seconds,
            "infer_seconds": infer_seconds,
            "realtime_factor": audio_seconds / infer_seconds,
            "logits_shape": tuple(logits.shape),
            "embeddings_shape": tuple(embeddings.shape),
            "decision": decision.retention_reason,
            "max_bio_label": decision.max_bio_label,
            "max_bio_logit": decision.max_bio_logit,
            "max_perch_label": decision.max_perch_label,
            "max_perch_logit": decision.max_perch_logit,
        }
    )

full_infer_seconds = time.perf_counter() - full_start
inference_df = pd.DataFrame(inference_rows)
inference_df

In [ ]:
audio_seconds_total = sum(row["audio_seconds"] for row in inference_rows)
steady_state_df = inference_df.iloc[1:] if len(inference_df) > 1 else inference_df

summary = {
    "buffers_processed": len(buffers),
    "perch_frames_processed": len(buffers) * 3,
    "audio_seconds_processed": audio_seconds_total,
    "total_inference_seconds": full_infer_seconds,
    "overall_realtime_factor": audio_seconds_total / full_infer_seconds,
    "mean_seconds_per_buffer": float(inference_df["infer_seconds"].mean()),
    "mean_seconds_per_buffer_excluding_first": float(steady_state_df["infer_seconds"].mean()),
    "mean_realtime_factor_excluding_first": float(steady_state_df["realtime_factor"].mean()),
}

pd.DataFrame([summary]).T.rename(columns={0: "value"})

In [ ]:
# A compact visual summary using pandas' Styler rather than an extra plotting dependency.
(
    inference_df[["buffer_index", "infer_seconds", "realtime_factor", "decision", "max_bio_label", "max_bio_logit", "max_perch_label"]]
    .style
    .format({"infer_seconds": "{:.3f}", "realtime_factor": "{:.1f}", "max_bio_logit": "{:.3f}"})
    .bar(subset=["infer_seconds"], color="#9ecae1")
    .bar(subset=["realtime_factor"], color="#a1d99b")
)

## 7. Inspect The Most Interesting Buffer

To keep the lesson concrete, we pick the buffer with the strongest configured biological label score. This should usually land near one of the petrel calls in the teaching clip.

In [ ]:
interesting_index = int(inference_df["max_bio_logit"].astype(float).idxmax())
interesting = results[interesting_index]
print("Selected buffer index:", interesting["buffer"].file_buffer_index)
print("Selected decision:", interesting["decision"].retention_reason)
print("Selected max bio:", interesting["decision"].max_bio_label, interesting["decision"].max_bio_logit)

Audio(
    interesting["buffer"].source_audio,
    rate=interesting["buffer"].source_sample_rate,
)

In [ ]:
score_rows = []
for score in interesting["frame_scores"]:
    score_rows.append(
        {
            "segment_index": score.segment_index,
            "max_noise_label": score.max_noise_label,
            "max_noise_logit": score.max_noise_logit,
            "max_bio_label": score.max_bio_label,
            "max_bio_logit": score.max_bio_logit,
            "max_perch_label": score.max_perch_label,
            "max_perch_logit": score.max_perch_logit,
        }
    )

pd.DataFrame(score_rows)

In [ ]:
nz_rows = []
for score in interesting["frame_scores"]:
    for rank, bird in enumerate(score.top_nz_birds, start=1):
        nz_rows.append({"segment_index": score.segment_index, "rank": rank, **bird})

pd.DataFrame(nz_rows)

## 8. Apply The First-Pass Gate

`decide_buffer()` converts frame scores into one buffer-level decision.

Current Phase 1 rule:

- `bio_hit` if any frame has a configured biological label score above `bio_threshold`.
- Otherwise `validation_sample` every `validation_sample_interval` dropped buffers.
- Otherwise `dropped`.

The gate also prepares compact JSON fields for database storage.

In [ ]:
decision = interesting["decision"]

print("Retention reason:", decision.retention_reason)
print("Audio saved?:", decision.audio_saved)
print("Max biological label:", decision.max_bio_label)
print("Max biological logit:", decision.max_bio_logit)
print("Max overall Perch label:", decision.max_perch_label)
print("Max overall Perch logit:", decision.max_perch_logit)

In [ ]:
pd.DataFrame(json.loads(decision.noise_logits))

In [ ]:
pd.json_normalize(json.loads(decision.nz_bird_logits), record_path="top_3", meta="segment_index")

## 9. Example: Save Retained `.flac` Audio

The real loop saves audio only when `decision.audio_saved` is true. This keeps full audio storage under control.

The example below writes retained teaching clips to `edge_node_mock/data/notebook_retained_audio/`, which is ignored by Git.

In [ ]:
example_retained_dir = REPO_ROOT / "edge_node_mock" / "data" / "notebook_retained_audio"

saved_paths = []
for item in results:
    buffer = item["buffer"]
    decision = item["decision"]
    if decision.audio_saved:
        saved_path = save_retained_audio(
            buffer,
            example_retained_dir,
            str(config["device_id"]),
            decision.retention_reason,
        )
        item["decision"] = type(decision)(**{**decision.__dict__, "filepath": saved_path})
        saved_paths.append(saved_path)

print("Retained FLAC files saved:", len(saved_paths))
for path in saved_paths:
    print(path)

## 10. Optional: Insert The Full Teaching Clip Into A Scratch Edge Database

The production script writes to the configured edge database. For teaching, this cell creates a scratch notebook database so you can inspect the schema without altering your main Phase 1 run.

The database and retained audio directory both live under `edge_node_mock/data/notebook_demo/`, which is ignored by Git.

In [ ]:
scratch_dir = REPO_ROOT / "edge_node_mock" / "data" / "notebook_demo"
scratch_dir.mkdir(parents=True, exist_ok=True)

scratch_config = copy.deepcopy(config)
scratch_config["edge_db_path"] = str(scratch_dir / "edge_notebook.sqlite")
scratch_config["retained_audio_dir"] = str(scratch_dir / "retained_audio")

scratch_config_path = scratch_dir / "edge_config.notebook.yaml"
scratch_config_path.write_text(yaml.safe_dump(scratch_config, sort_keys=False), encoding="utf-8")

db_path = init_edge_db(scratch_config_path, reset=True)
print("Scratch database:", db_path)

In [ ]:
with sqlite3.connect(db_path) as conn:
    conn.execute("PRAGMA foreign_keys=ON;")
    inserted_ids = []
    for item in results:
        buffer_id = insert_buffer_event(
            conn,
            config=scratch_config,
            buffer=item["buffer"],
            decision=item["decision"],
            embeddings=item["embeddings"],
        )
        inserted_ids.append(buffer_id)
    conn.commit()

print("Inserted buffer IDs:", inserted_ids)

In [ ]:
with sqlite3.connect(db_path) as conn:
    counts = {
        "buffer_events": conn.execute("SELECT COUNT(*) FROM buffer_events;").fetchone()[0],
        "embedding_segments": conn.execute("SELECT COUNT(*) FROM embedding_segments;").fetchone()[0],
    }
    vector_table = dict(conn.execute("SELECT key, value FROM schema_metadata;").fetchall())["vector_table"]

    # sqlite-vec virtual tables need the extension loaded before querying them.
    if vector_table == "perch_vectors":
        load_sqlite_vec(conn)

    counts[vector_table] = conn.execute(f"SELECT COUNT(*) FROM {vector_table};").fetchone()[0]
    rows = conn.execute(
        """
        SELECT buffer_id, retention_reason, audio_saved, max_bio_label,
               ROUND(max_bio_logit, 3), sync_status, filepath
        FROM buffer_events
        ORDER BY buffer_id;
        """
    ).fetchall()

print(counts)
pd.DataFrame(
    rows,
    columns=["buffer_id", "retention_reason", "audio_saved", "max_bio_label", "max_bio_logit", "sync_status", "filepath"],
)

## 11. What The Full Script Adds

This notebook walked through the teaching clip manually. The full `bio_capture_loop.py` wraps these same pieces in a loop:

```text
load config
initialize edge database
load labels and model
for each 15-second audio buffer:
    run Perch inference
    score the three frames
    make a retention decision
    save FLAC if needed
    insert metadata, segments, and vectors
```

The normal bounded run command for your configured raw dataset is:

```bash
.venv/bin/python edge_node_mock/src/bio_capture_loop.py --config edge_node_mock/config/edge_config.local.yaml --iterations 3
```

For teaching, the important idea is that the pipeline is small, inspectable steps. Each step has a concrete object: a buffer, logits, embeddings, frame scores, a decision, a retained audio path, and finally database rows.